# 03 – Construcción del Workbook de Seguimiento

# Objetivo

Construir de forma ordenada las tablas necesarias para el sistema de seguimiento del proyecto, partiendo del modelo definido en el Notebook 02.

En esta primera parte construiremos:
- Activity Master
- Project Tracking (Nivel 1)

# Setup

In [1]:
from pathlib import Path
import openpyxl
import pandas as pd
import numpy as np
import re
from datetime import datetime

BASE_DIR = Path.cwd()
DATA_ORIGINAL = BASE_DIR / "Data_Original"
EXCEL_FILE = DATA_ORIGINAL / "C2011-13 Wind Farm.xlsx"
OUTPUT_DIR = BASE_DIR / "03_Outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Rutas configuradas")
print("Archivo Excel existe:", EXCEL_FILE.exists())

# Carga archivo excel original

In [2]:
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)
print("Archivo cargado correctamente")
print(f"Número de hojas: {len(wb.sheetnames)}")

# PARTE A - Activity Master - Project Tracking

## Construir Activity Master

In [3]:
ws = wb["Baseline Schedule"]

rows = []
for row in range(3, ws.max_row + 1):
    id_ = ws.cell(row, 1).value
    if id_ is None:
        continue

    rows.append({
        "ID": id_,
        "Name": ws.cell(row, 2).value,
        "WBS": ws.cell(row, 3).value,
        "Predecessors": ws.cell(row, 4).value,
        "Successors": ws.cell(row, 5).value,
        "Baseline_Start": ws.cell(row, 6).value,
        "Baseline_End": ws.cell(row, 7).value,
        "Duration_Plan": ws.cell(row, 8).value,
        "Total_Cost_Plan": ws.cell(row, 14).value,
        "Fixed_Cost": ws.cell(row, 11).value,
        "Variable_Cost": ws.cell(row, 13).value,
        "Resource_Cost": ws.cell(row, 10).value,
    })

df_master = pd.DataFrame(rows)

# Limpieza de duración
def parse_duration(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    match = re.match(r"([\d\.]+)\s*d?", str(value).strip().lower())
    return float(match.group(1)) if match else np.nan

df_master["Duration_Plan_Days"] = df_master["Duration_Plan"].apply(parse_duration)
df_master["WBS_Level"] = df_master["WBS"].astype(str).str.count(r"\.") + 1

print(f"Activity Master creada: {len(df_master)} actividades")
df_master.head(10)

## Comprobar Activity Master

In [6]:
root = df_master[df_master["ID"] == 0].iloc[0]

print("=== Comprobación Activity Master ===")
print(f"Actividades totales : {len(df_master)}")
print(f"Duración planificada: {root['Duration_Plan_Days']} días")
print(f"Coste planificado   : {root['Total_Cost_Plan']:,.0f} €")
print()
print(df_master[["ID", "WBS", "Name", "Duration_Plan_Days", "Total_Cost_Plan"]].head(8))

## Construir Project Tracking (Nivel 1)

In [8]:
ws = wb["Tracking Overview"]

records = []
for row in range(3, ws.max_row + 1):
    tp_name = ws.cell(row, 1).value
    if tp_name is None:
        continue

    records.append({
        "TP": int(tp_name) if str(tp_name).isdigit() else tp_name,
        "Start_Tracking_Period": ws.cell(row, 2).value,
        "Status_Date": ws.cell(row, 3).value,
        "PV": ws.cell(row, 4).value,
        "EV": ws.cell(row, 5).value,
        "AC": ws.cell(row, 6).value,
        "ES": ws.cell(row, 7).value,
        "SV": ws.cell(row, 8).value,
        "SPI": ws.cell(row, 9).value,
        "CV": ws.cell(row, 10).value,
        "CPI": ws.cell(row, 11).value,
        "SV_t": ws.cell(row, 12).value,
        "SPI_t": ws.cell(row, 13).value,
        "p_factor": ws.cell(row, 14).value,
    })

df_project_tracking = pd.DataFrame(records)

# Convertir a numérico las columnas clave
for col in ["PV", "EV", "AC", "SV", "SPI", "CV", "CPI", "SPI_t", "p_factor"]:
    df_project_tracking[col] = pd.to_numeric(df_project_tracking[col], errors="coerce")

print(f"Project Tracking creado: {len(df_project_tracking)} periodos")
df_project_tracking.tail(8)

## Comprobar Project Tracking

In [10]:
print("=== Comprobación Project Tracking ===")
print(f"Número de periodos: {len(df_project_tracking)}")
print()
print("Primer periodo:")
print(df_project_tracking.iloc[0][["TP", "Status_Date", "PV", "EV", "AC", "SPI", "CPI"]])
print()
print("Último periodo:")
print(df_project_tracking.iloc[-1][["TP", "Status_Date", "PV", "EV", "AC", "SPI", "CPI"]])

## Guardar resultados intermedios (Parte A)

In [12]:
df_master.to_excel(OUTPUT_DIR / "Activity_Master.xlsx", index=False)
df_project_tracking.to_excel(OUTPUT_DIR / "Project_Tracking.xlsx", index=False)

print("Archivos intermedios guardados en 03_Outputs:")
print(" - Activity_Master.xlsx")
print(" - Project_Tracking.xlsx")

# PARTE B - Activity Tracking

## Activity Tracking

In [13]:
def read_tracking_period(wb, sheet_name, tp_number):
    """
    Lee una hoja de Tracking Period y devuelve un DataFrame
    con el estado de las actividades en ese periodo.
    """
    ws = wb[sheet_name]
    
    # La fecha de status está en la celda C1 (fila 1, columna 3)
    status_date = ws.cell(1, 3).value
    
    rows = []
    for row in range(5, ws.max_row + 1):
        activity_id = ws.cell(row, 1).value
        if activity_id is None:
            continue
            
        rows.append({
            "TP": tp_number,
            "Status_Date": status_date,
            "Activity_ID": activity_id,
            "Actual_Start": ws.cell(row, 12).value,
            "Actual_Duration": ws.cell(row, 13).value,
            "Actual_Cost": ws.cell(row, 19).value,
            "Pct_Complete": ws.cell(row, 21).value,
            "EV": ws.cell(row, 23).value,
            "PV": ws.cell(row, 24).value,
        })
    
    return pd.DataFrame(rows)

In [14]:
# Probamos primero con el primer y el último periodo para validar
df_tp1 = read_tracking_period(wb, "Project Control - TP1", 1)
df_tp120 = read_tracking_period(wb, "TP120", 120)

print("TP1 – primeras filas:")
print(df_tp1.head(6))
print()
print("TP120 – primeras filas:")
print(df_tp120.head(6))
print()
print(f"TP1 filas: {len(df_tp1)} | TP120 filas: {len(df_tp120)}")

In [15]:
# Construir Activity Tracking completo (los 120 periodos)

In [18]:
# Lista de hojas de tracking
# TP1 tiene nombre especial, el resto es TP2 ... TP120
tracking_sheets = []
tracking_sheets.append(("Project Control - TP1", 1))

for i in range(2, 121):
    tracking_sheets.append((f"TP{i}", i))

print(f"Total de periodos a procesar: {len(tracking_sheets)}")

# Construcción de la tabla
all_tracking = []

for sheet_name, tp_number in tracking_sheets:
    df_tp = read_tracking_period(wb, sheet_name, tp_number)
    all_tracking.append(df_tp)
    
    if tp_number % 20 == 0 or tp_number == 1:
        print(f"  Procesado TP{tp_number} – {len(df_tp)} actividades")

df_activity_tracking = pd.concat(all_tracking, ignore_index=True)

print()
print(f"Activity Tracking creado: {len(df_activity_tracking):,} filas")
print(f"Periodos únicos: {df_activity_tracking['TP'].nunique()}")
print(f"Actividades únicas: {df_activity_tracking['Activity_ID'].nunique()}")

## Limpieza básica de Activity Tracking

In [20]:
# Convertir duración a días numéricos
df_activity_tracking["Actual_Duration_Days"] = df_activity_tracking["Actual_Duration"].apply(parse_duration)

# Asegurar tipos numéricos
for col in ["Actual_Cost", "Pct_Complete", "EV", "PV"]:
    df_activity_tracking[col] = pd.to_numeric(df_activity_tracking[col], errors="coerce")

print("Limpieza aplicada")
print()
print(df_activity_tracking.head(8))

## Comprobación final de Activity Tracking

In [22]:
print("=== Comprobación Activity Tracking ===")
print(f"Filas totales        : {len(df_activity_tracking):,}")
print(f"Periodos             : {df_activity_tracking['TP'].nunique()}")
print(f"Actividades          : {df_activity_tracking['Activity_ID'].nunique()}")
print()

# Comprobación del proyecto raíz (Activity_ID = 0) en el primer y último periodo
root_first = df_activity_tracking[(df_activity_tracking["Activity_ID"] == 0) & (df_activity_tracking["TP"] == 1)].iloc[0]
root_last  = df_activity_tracking[(df_activity_tracking["Activity_ID"] == 0) & (df_activity_tracking["TP"] == 120)].iloc[0]

print("Proyecto raíz (ID=0) – Primer periodo:")
print(f"  Status Date     : {root_first['Status_Date']}")
print(f"  Actual Duration : {root_first['Actual_Duration']}")
print(f"  Actual Cost     : {root_first['Actual_Cost']:,.0f}")
print(f"  % Complete      : {root_first['Pct_Complete']:.4f}")
print()
print("Proyecto raíz (ID=0) – Último periodo:")
print(f"  Status Date     : {root_last['Status_Date']}")
print(f"  Actual Duration : {root_last['Actual_Duration']}")
print(f"  Actual Cost     : {root_last['Actual_Cost']:,.0f}")
print(f"  % Complete      : {root_last['Pct_Complete']:.4f}")

## Guardar Activity Tracking

In [24]:
df_activity_tracking.to_excel(OUTPUT_DIR / "Activity_Tracking.xlsx", index=False)
print("Guardado: 03_Outputs/Activity_Tracking.xlsx")

# PARTE C - GENERACION EXCEL FINAL

In [26]:
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill, NamedStyle
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter

output_file = OUTPUT_DIR / "Project_Control_Workbook.xlsx"

# Crear libro
wb_out = openpyxl.Workbook()

# -------------------------------------------------
# Hoja 0: Documentation
# -------------------------------------------------
ws_doc = wb_out.active
ws_doc.title = "00_Documentation"

doc_content = [
    ["PROJECT CONTROL WORKBOOK"],
    [""],
    ["Project:", "C2011-13 Wind Farm"],
    ["Source:", "DSLIB – Dynamic Scheduling Empirical Project Library (Ghent University)"],
    ["Generated on:", datetime.now().strftime("%Y-%m-%d %H:%M")],
    [""],
    ["Purpose"],
    ["This workbook provides a clean and structured project control system based on the original project data."],
    ["It is designed to support project monitoring and simulation of project management over time."],
    [""],
    ["Sheets included:"],
    ["01_Activity_Master", "Stable definition of all project activities (WBS, baseline, costs...)"],
    ["02_Project_Tracking", "Project-level performance over 120 tracking periods (EVM indicators)"],
    ["03_Activity_Tracking", "Detailed status of each activity in each tracking period"],
    [""],
    ["How to use this workbook"],
    ["1. Start with 01_Activity_Master to understand the project structure."],
    ["2. Use 02_Project_Tracking to analyse overall project performance over time."],
    ["3. Use 03_Activity_Tracking for detailed activity-level follow-up."],
    [""],
    ["This structure allows building dashboards, alerts and management simulations."],
]

for r_idx, row in enumerate(doc_content, 1):
    for c_idx, value in enumerate(row, 1):
        ws_doc.cell(row=r_idx, column=c_idx, value=value)

ws_doc.column_dimensions['A'].width = 25
ws_doc.column_dimensions['B'].width = 90

# -------------------------------------------------
# Hoja 1: Activity Master
# -------------------------------------------------
ws_master = wb_out.create_sheet("01_Activity_Master")

for r in dataframe_to_rows(df_master, index=False, header=True):
    ws_master.append(r)

# -------------------------------------------------
# Hoja 2: Project Tracking
# -------------------------------------------------
ws_proj = wb_out.create_sheet("02_Project_Tracking")

for r in dataframe_to_rows(df_project_tracking, index=False, header=True):
    ws_proj.append(r)

# -------------------------------------------------
# Hoja 3: Activity Tracking
# -------------------------------------------------
ws_act = wb_out.create_sheet("03_Activity_Tracking")

for r in dataframe_to_rows(df_activity_tracking, index=False, header=True):
    ws_act.append(r)

# Guardar
wb_out.save(output_file)
print(f"Workbook generado correctamente:")
print(f"→ {output_file}")
print()
print("Hojas creadas:")
print("  - 00_Documentation")
print("  - 01_Activity_Master")
print("  - 02_Project_Tracking")
print("  - 03_Activity_Tracking")

# RESUMEN FINAL DEL CUADERNO

In [27]:
print("=" * 60)
print("RESUMEN FINAL – PROJECT CONTROL WORKBOOK")
print("=" * 60)
print(f"Activity Master      : {len(df_master):,} actividades")
print(f"Project Tracking     : {len(df_project_tracking):,} periodos")
print(f"Activity Tracking    : {len(df_activity_tracking):,} registros")
print()
print(f"Archivo generado en  : {output_file}")
print("=" * 60)